# Phase 2 — Structured LLM Extraction

**Project:** VIGILOX Document Intelligence
**Phase Status:** Complete ✅

## 1. Purpose

Phase 2 converts raw OCR output into structured document data.

Phase 1 produced:

* OCR text
* OCR confidence
* Bounding boxes
* Line ordering

Phase 2 turns that unstructured OCR output into fields that can be used by the application.

```text
Document Image
      ↓
PaddleOCR
      ↓
Raw OCR Lines
      ↓
Groq LLM
      ↓
Structured Document Data
```

The goal was not to let the LLM freely summarize the document. It was used as a **strict information extraction layer**.

---

## 2. Groq API

Groq was used as the LLM provider for structured extraction.

The extraction service sends OCR lines to the Groq model together with:

* A unique line ID
* OCR text
* Bounding-box information

The model used during this phase was:

```text
openai/gpt-oss-20b
```

Groq structured output was used so the response followed a predefined JSON schema rather than returning free-form text.

---

## 3. Pydantic Schema Validation

Pydantic was used to define and validate the required document structure.

The schema ensures that the LLM must return the expected fields and cannot freely add arbitrary properties.

The extracted document contains:

| Field            | Purpose                   |
| ---------------- | ------------------------- |
| `document_type`  | Type of document          |
| `full_name`      | Person's name             |
| `licence_number` | Security/licence number   |
| `id_number`      | National or document ID   |
| `expiry_date`    | Expiration date           |
| `date_of_birth`  | DOB                       |
| `issue_date`     | Explicit issue/print date |
| `issuer`         | Issuing authority         |

Supported document classifications are:

```text
sia_badge
id_card
guard_license
unknown
```

Missing information is returned as `null` rather than guessed.

---

## 4. Evidence Mapping with Source Line IDs

An important design decision was to make every extracted field traceable back to OCR evidence.

Each OCR line is assigned an identifier:

```text
L0
L1
L2
L3
...
```

Example:

```text
L4 → EXPIRES
L5 → 24 MAR 2021
```

The structured extraction can therefore return:

```text
expiry_date = 2021-03-24
source_line_ids = [L4, L5]
```

This creates a direct relationship between:

```text
Extracted Field
      ↓
Source OCR Lines
      ↓
Original Document Evidence
```

This evidence mapping later became the foundation for Phase 3 validation.

---

## 5. Why String Line IDs Were Used

Initially, numeric line references caused an LLM reliability problem.

For example, separate lines:

```text
4
5
```

were incorrectly returned as:

```text
45
```

Instead of:

```text
[4, 5]
```

The evidence identifier format was therefore changed to:

```text
L4
L5
```

The expected output became:

```text
["L4", "L5"]
```

This made references clearer and easier to validate deterministically.

---

## 6. Key Extraction Rules

The LLM was constrained with strict extraction rules.

The most important rules were:

* Never invent or guess fields.
* Every extracted value must be supported by OCR evidence.
* Missing fields must be returned as `null`.
* Licence and ID numbers should be preserved as recognized by OCR.
* Dates should only be normalized when clearly supported.
* Ambiguous dates should not be guessed.
* An unlabeled date must not automatically be classified as an issue date.
* Duplicate representations of the same date must not be treated as separate fields.
* Relevant label/context lines should be included in `source_line_ids`.
* Source IDs must exactly match OCR input IDs.
* Source line IDs must never be concatenated or invented.
* Confidence scores must not be generated by the LLM.

---

## 7. Document Classification Rules

Basic document classification was also included in the extraction prompt.

For example:

* `Security Industry Authority` together with licence information → `sia_badge`
* Generic private/security licence → `guard_license`
* National identity document → `id_card`
* Insufficient evidence → `unknown`

This classification is part of the structured LLM output, rather than a separate deterministic classifier.

---

## 8. Main Phase 2 Result

For the SIA badge, the final integrated extraction correctly produced:

```text
Document Type:
sia_badge

Full Name:
M.GREEN
Evidence: L6

Licence Number:
1099 4265 1706 9065
Evidence: L0

Expiry Date:
2021-03-24
Evidence: L4, L5

Issuer:
Security Industry Authority
Evidence: L3

ID Number:
null

Date of Birth:
null

Issue Date:
null
```

The result demonstrated that raw OCR could successfully be transformed into structured, traceable document information.

---

## 9. Important Findings

### Structured output is better than free-form LLM responses

Using a strict schema made the output predictable and easier to process programmatically.

### Schema compliance does not guarantee semantic correctness

Even when an LLM returns perfectly valid JSON, the extracted information or evidence references can still be wrong.

Therefore:

> Valid JSON is not the same as validated document information.

This observation directly motivated Phase 3.

### Evidence provenance is essential

Returning only:

```text
expiry_date = 2021-03-24
```

is not sufficient for a trustworthy document-intelligence system.

Returning:

```text
expiry_date = 2021-03-24
evidence = L4, L5
```

makes the extraction auditable and independently verifiable.

### LLMs should not generate confidence scores

OCR confidence comes from the OCR model. The LLM's role in this phase is structured interpretation, not assigning unsupported confidence values.

---

## 10. Phase 2 Final Conclusion

Phase 2 successfully added a structured LLM extraction layer on top of PaddleOCR.

The system can now convert raw OCR lines into a strict document schema while preserving links back to the supporting OCR evidence.

The final Phase 2 pipeline is:

```text
OCR Text + Bounding Boxes
        ↓
Line IDs
        ↓
Groq Structured Extraction
        ↓
Pydantic Schema Validation
        ↓
Structured Fields + Evidence References
```

**Phase 2 is complete.** The remaining reliability problem at this stage was verifying whether the referenced OCR evidence actually supported each extracted value, which became the focus of Phase 3.
